# Day 67: RNNs & LSTMs — Understanding Sequences
## Time Series, Text, and Memory in Neural Networks

---

# PART 1: THEORY

## 1. The Problem with Sequences

Normal neural networks treat each input independently. But sequences have ORDER:

- **Stock prices:** Yesterday's price matters for today
- **Text:** "not good" vs "good" — word order changes meaning completely
- **Speech:** Sounds form words, words form sentences
- **Video:** Frames form actions over time

A Dense network sees "not good" and "good" as just two words each. It misses the ORDER.

## 2. Recurrent Neural Networks (RNNs)

An RNN has a **loop** — output from the previous step feeds into the next:

```
h_t = activation(W * x_t + U * h_{t-1} + b)
```

At each time step, the RNN sees:
1. Current input (x_t)
2. Hidden state from previous step (h_{t-1}) — the "memory"

**The hidden state is the memory.** It carries information from earlier time steps forward.

## 3. The Vanishing Gradient Problem

During backpropagation through time, gradients get multiplied at each step. If gradients < 1, they shrink exponentially — early time steps receive almost no learning signal.

**Result:** RNNs can only remember ~10-20 time steps back. Information from long ago "vanishes."

## 4. LSTM (Long Short-Term Memory)

LSTM solves vanishing gradients with a **gating mechanism** and a **cell state**:

| Gate | What It Does | Analogy |
|------|-------------|---------|
| **Forget Gate** | What old info to discard? | Delete button |
| **Input Gate** | What new info to store? | Save button |
| **Output Gate** | What to output now? | Read button |

The **cell state** acts like a conveyor belt — information can travel long distances without fading. This lets LSTMs remember information from hundreds of time steps ago.

## 5. GRU (Gated Recurrent Unit)

A simplified LSTM with only 2 gates (reset and update). Fewer parameters, often performs similarly. Use when you want a simpler model.

---

# PART 2: PRACTICAL

## 6. Part A: Time Series Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
# Generate sine wave data
t = np.linspace(0, 100, 1000)
data = np.sin(t) + 0.1 * np.random.randn(1000)

plt.figure(figsize=(12, 4))
plt.plot(t[:300], data[:300], 'b-', linewidth=1)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Sine Wave with Noise')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Create sequences: use last 50 steps to predict next step
def create_sequences(data, seq_len=50):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(data)
X = X.reshape(-1, 50, 1)  # (samples, timesteps, features)
y = y.reshape(-1, 1)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Each sample: 50 time steps -> predict 1 value")


In [ ]:
# Build and compare: SimpleRNN vs LSTM vs GRU
results = {}
for name, layer_class in [('SimpleRNN', layers.SimpleRNN), ('LSTM', layers.LSTM), ('GRU', layers.GRU)]:
    m = keras.Sequential([
        layer_class(64, input_shape=(50, 1), return_sequences=True),
        layer_class(32),
        layers.Dense(1)
    ])
    m.compile(optimizer='adam', loss='mse')
    history = m.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_data=(X_test, y_test), verbose=0)
    pred = m.predict(X_test, verbose=0)
    mse = np.mean((y_test - pred) ** 2)
    results[name] = {'model': m, 'mse': mse, 'pred': pred}
    print(f"{name:12s} -> MSE: {mse:.5f}")

# Plot predictions
plt.figure(figsize=(14, 6))
plt.plot(y_test[:150], 'k-', label='Actual', linewidth=2, alpha=0.7)
for name, color in [('SimpleRNN', 'blue'), ('LSTM', 'red'), ('GRU', 'green')]:
    plt.plot(results[name]['pred'][:150], '--', label=f'{name} (MSE={results[name]["mse"]:.4f})', color=color)
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Time Series Prediction — RNN vs LSTM vs GRU')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 7. Part B: Sentiment Analysis with LSTM

In [ ]:
# Load IMDB movie reviews
vocab_size = 10000
max_len = 200

(X_train_text, y_train_text), (X_test_text, y_test_text) = keras.datasets.imdb.load_data(num_words=vocab_size)

# Pad to same length
X_train_pad = keras.preprocessing.sequence.pad_sequences(X_train_text, maxlen=max_len)
X_test_pad = keras.preprocessing.sequence.pad_sequences(X_test_text, maxlen=max_len)

print(f"Training reviews: {X_train_pad.shape}")
print(f"Test reviews: {X_test_pad.shape}")
print(f"Vocabulary size: {vocab_size}")
print(f"\nLabels: 0=Negative, 1=Positive")
print(f"Class balance (train): {y_train_text.mean():.1%} positive reviews")

# Show a decoded review
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v+3: k for k, v in word_index.items()}
reverse_word_index[1], reverse_word_index[2], reverse_word_index[3] = '<START>', '<UNK>', '<PAD>'
decoded = ' '.join([reverse_word_index.get(i, '?') for i in X_train_text[0]])
print(f"\nExample review:\n{decoded[:200]}...")
print(f"Sentiment: {'Positive' if y_train_text[0] == 1 else 'Negative'}")


In [ ]:
# Build sentiment LSTM
sentiment_model = keras.Sequential([
    # Embedding: Convert word indices -> dense vectors
    layers.Embedding(vocab_size, 128, input_length=max_len),

    # Two stacked LSTMs
    layers.LSTM(64, return_sequences=True, dropout=0.3),
    layers.LSTM(32, dropout=0.3),

    # Classifier
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

sentiment_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
sentiment_model.summary()


In [ ]:
# Train
early_stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history = sentiment_model.fit(X_train_pad, y_train_text, epochs=10, batch_size=64,
                               validation_split=0.2, callbacks=[early_stop], verbose=1)

test_loss, test_acc = sentiment_model.evaluate(X_test_pad, y_test_text, verbose=0)
print(f"\nSentiment Analysis Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Predict on custom reviews
def predict_sentiment(text):
    # Tokenize (simple word index lookup)
    words = text.lower().split()
    tokens = [word_index.get(w, 2) + 3 for w in words]  # 2 = <UNK>, +3 for reserved tokens
    padded = keras.preprocessing.sequence.pad_sequences([tokens], maxlen=max_len)
    prob = sentiment_model.predict(padded, verbose=0)[0][0]
    return prob, 'Positive' if prob > 0.5 else 'Negative'

reviews = [
    "this movie was absolutely fantastic and I loved every minute of it",
    "terrible waste of time the acting was awful and the plot made no sense",
    "it was okay not great but not terrible either just average",
]

for review in reviews:
    prob, sentiment = predict_sentiment(review)
    print(f"\nReview: '{review}'")
    print(f"Sentiment: {sentiment} ({prob:.1%} confidence)")


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Compare Embedding dimension size
for emb_dim in [32, 64, 128, 256]:
    m = keras.Sequential([
        layers.Embedding(vocab_size, emb_dim, input_length=max_len),
        layers.GlobalAveragePooling1D(),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_train_pad[:5000], y_train_text[:5000], epochs=3, validation_split=0.2, verbose=0)
    _, acc = m.evaluate(X_test_pad[:1000], y_test_text[:1000], verbose=0)
    print(f"Embedding dim {emb_dim:4d} -> Test Acc: {acc:.3f}")


In [ ]:
# Exercise 2: Try 1D CNN for text (faster alternative to LSTM)
cnn_text = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.Conv1D(128, 5, activation='relu'),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation='sigmoid')
])
cnn_text.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_text.fit(X_train_pad[:5000], y_train_text[:5000], epochs=3, validation_split=0.2, verbose=0)
_, cnn_acc = cnn_text.evaluate(X_test_pad[:1000], y_test_text[:1000], verbose=0)
print(f"1D CNN for text -> Test Acc: {cnn_acc:.3f}")
print("1D CNNs are often as good as LSTMs for text and MUCH faster!")


## Key Takeaways

- **RNNs** process sequences by passing hidden state through time
- **LSTM** solves vanishing gradients with gates (forget, input, output)
- **GRU** is a simpler alternative to LSTM
- **Time series** and **text** are the two main sequence applications
- **Embedding** layer converts word indices to dense vectors
- LSTM with 2 stacked layers + dropout is a strong text model
- 1D CNNs are a fast alternative for text classification

**Tomorrow:** NLP with Deep Learning — word embeddings, transformers, and modern NLP!